# CIDA Role Pre-Training
## Цель: научить агентов КТО ОНИ, а не ЧТО КЛАССИФИЦИРОВАТЬ

```
Обычное предобучение:  агент → задача → правильный ответ
Ролевое предобучение:  агент → любая задача → поведение своей роли
```

## Архитектура
```
Текст/Данные
     ↓
MiniLM (заморожен, 22M) — универсальные sentence embeddings
     ↓  d=384
RoleIdentityLayer — якорь роли (обучается)
     ↓
CIDA Plugin (4 агента) — объект предобучения
     ↓
Ролевые лоссы (не классификация)
```

## 3 стадии
| Стадия | Данные | Что учат агенты |
|---|---|---|
| 1 | Kialo + ArgQ (дебаты) | Своё место в споре |
| 2 | ChaosNLI (неопределённость) | Когда быть уверенным, когда нет |
| 3 | Cross-domain mix | Роль одинакова в любом домене |


## 0. Setup

In [ ]:
!git clone https://github.com/Kairatzh/CIDA-CheXpert.git 2>/dev/null || (cd CIDA-CheXpert && git pull)
%cd CIDA-CheXpert
!pip install torchdiffeq datasets transformers sentence-transformers \
             scikit-learn requests -q

import sys
sys.path.insert(0, '/content/CIDA-CheXpert')
from cida_plugin import CIDAPlugin, CIDAPluginConfig
from cida_plugin.agent import RoleNames
from cida_plugin.diagnostics import DebateDiagnostics
print('CIDA OK')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np, os, time, warnings, math, json
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
IS_AMP  = DEVICE == 'cuda'
DTYPE   = torch.bfloat16 if (DEVICE=='cuda' and
          torch.cuda.get_device_capability()[0]>=8) else torch.float16

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark        = True

torch.manual_seed(42); np.random.seed(42)
os.makedirs('ckpt_role', exist_ok=True)
os.makedirs('results_role', exist_ok=True)
HISTORY = []

## 1. Encoder — Frozen MiniLM

MiniLM заморожен полностью. Он только переводит текст в вектор.
Агенты не учатся понимать текст — они учатся рассуждать над готовыми векторами.
При дообучении на CV заменяешь MiniLM на ResNet — CIDA не меняется.

In [ ]:
print('Loading MiniLM (frozen)...')
_minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
_minilm.to(DEVICE)
for p in _minilm.parameters():
    p.requires_grad = False

D_EMB = 384  # MiniLM output dim

@torch.no_grad()
def embed_texts(texts, batch_size=512):
    """Encode list of strings → (N, 384) tensor. Cached for speed."""
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        vecs  = _minilm.encode(batch, convert_to_tensor=True,
                               device=DEVICE, show_progress_bar=False)
        all_vecs.append(vecs.cpu())
    return torch.cat(all_vecs)

# Smoke test
test_emb = embed_texts(['The patient has pneumonia.', 'No pathology detected.'])
print(f'Embedding shape: {test_emb.shape}')  # (2, 384)
print(f'MiniLM params: {sum(p.numel() for p in _minilm.parameters())/1e6:.1f}M (all frozen)')

## 2. RoleIdentityLayer — ДНК агента

Это ключевая новая идея. Каждый агент имеет обучаемый вектор-якорь.
Он добавляется к входу ДО того как агент начинает рассуждать.
Этот вектор сохраняется в checkpoint и переносится на любую задачу.

```
input_vec (384) + role_anchor_i (384) → agent_i начинает рассуждение
```

Якорь инициализируется не случайно, а через прототипные примеры роли:
- Прокурор: среднее embedding сильных аргументов 'за'
- Защитник: среднее embedding контраргументов  
- Скептик:  среднее embedding неопределённых утверждений
- Интегратор: нулевой вектор (без bias)

In [ ]:
class RoleIdentityLayer(nn.Module):
    """
    Learnable role anchors. One per agent.
    Initialized from prototypical examples, not random.
    """
    ROLE_PROTOTYPES = {
        0: [  # Prosecutor: strong 'pro' arguments
            'The evidence clearly demonstrates this is true.',
            'Multiple studies confirm this conclusion beyond doubt.',
            'This is definitively supported by the data.',
            'The facts unambiguously point to this being correct.',
            'There is overwhelming support for this position.',
        ],
        1: [  # Defender: counter-arguments
            'However, this argument fails to account for key exceptions.',
            'The opposing evidence suggests we should reconsider.',
            'This claim is not supported when examined carefully.',
            'There are significant flaws in this line of reasoning.',
            'Alternative explanations are equally or more plausible.',
        ],
        2: [  # Skeptic: uncertainty
            'It is unclear whether this conclusion is warranted.',
            'The evidence is mixed and difficult to interpret.',
            'We cannot be certain given the available information.',
            'This remains an open question with no clear answer.',
            'Further investigation is needed before drawing conclusions.',
        ],
        3: [  # Integrator: synthesis
            'Taking all perspectives into account, the most balanced view is.',
            'Weighing the evidence from all sides leads to this synthesis.',
            'Both arguments have merit and the truth likely lies between them.',
            'Integrating these viewpoints gives us a more complete picture.',
            'The consensus emerging from all perspectives suggests.',
        ],
    }

    def __init__(self, d_emb=D_EMB, n_agents=4):
        super().__init__()
        # Initialize anchors from prototypes
        anchors = []
        print('Initializing role anchors from prototypes...')
        for i in range(n_agents):
            proto_vecs = embed_texts(self.ROLE_PROTOTYPES[i])  # (5, 384)
            anchor     = proto_vecs.mean(0)                     # (384,)
            anchors.append(anchor)
            sim = F.cosine_similarity(anchor.unsqueeze(0),
                                      anchors[0].unsqueeze(0) if i > 0
                                      else anchor.unsqueeze(0)).item()
            print(f'  {RoleNames.get(i)}: anchor norm={anchor.norm():.3f}')
        # Stack and make learnable
        self.anchors = nn.Parameter(torch.stack(anchors))  # (M, d_emb)
        # Small projection to let anchors adapt
        self.proj = nn.Sequential(
            nn.Linear(d_emb, d_emb),
            nn.LayerNorm(d_emb),
        )

    def forward(self, x):
        """
        x: (B, d_emb) — input embedding
        Returns: (B, M, d_emb) — x + role_anchor for each agent
        """
        B = x.size(0)
        # Normalize anchors to unit sphere
        anchors_norm = F.normalize(self.anchors, p=2, dim=-1)  # (M, d)
        # Add role anchor to input
        x_exp     = x.unsqueeze(1).expand(-1, self.anchors.size(0), -1)  # (B, M, d)
        anchors_e = anchors_norm.unsqueeze(0).expand(B, -1, -1)           # (B, M, d)
        combined  = x_exp + 0.3 * anchors_e   # blend: 70% data, 30% identity
        return combined  # (B, M, d)

    def anchor_similarity_matrix(self):
        """How similar are anchors to each other? Should be low."""
        a = F.normalize(self.anchors, p=2, dim=-1)
        return (a @ a.T).detach().cpu()


role_layer = RoleIdentityLayer(D_EMB, 4).to(DEVICE)
print('\nAnchor similarity matrix (lower = more diverse):')
sim = role_layer.anchor_similarity_matrix()
print(sim.numpy().round(3))

## 3. CIDA Config

In [ ]:
def make_cfg(num_classes):
    return CIDAPluginConfig(
        d_input=D_EMB,
        d_hidden=256,
        d_message=256,
        num_agents=4,
        num_classes=num_classes,
        max_rounds=3,
        multi_label=False,
        num_attn_heads=8,
        role_prior_blend=0.40,
        lambda_cal=0.3,
        lambda_ac=0.1,
        min_disagreement=0.08,
        comm_dropout=0.15,
        use_perspective_projector=True,
        anonymous_messages=True,
        sparse_comm_k=2,
        use_dynamic_trust=True,
        trust_ema_gamma=0.9,
    )


class RoleCIDA(nn.Module):
    """
    Role-aware CIDA:
    1. Input embedding x (B, d)
    2. RoleIdentityLayer adds anchors → (B, M, d)
    3. Each agent_i gets x + anchor_i as its starting point
    4. CIDA deliberation runs
    """
    def __init__(self, num_classes=3):
        super().__init__()
        self.role_layer = RoleIdentityLayer(D_EMB, 4)
        self.plugin     = CIDAPlugin(make_cfg(num_classes))
        self.num_classes = num_classes

    def forward(self, x):
        """
        x: (B, D_EMB) — pre-computed embedding (MiniLM output)
        """
        # Role anchors applied BEFORE CIDA
        # We pass pooled=x (standard), CIDA will use its own perspective projector
        # The role_layer output is used to bias the initial state via role_prior_blend
        out = self.plugin(x)
        out['role_anchored'] = self.role_layer(x)  # (B, M, d) for loss computation
        return out


# Test
test_model = RoleCIDA(num_classes=3).to(DEVICE)
test_x = torch.randn(4, D_EMB).to(DEVICE)
test_out = test_model(test_x)
print(f'p_final: {test_out["p_final"].shape}')
print(f'b_all rounds: {len(test_out["b_all"])}')
total = sum(p.numel() for p in test_model.parameters())
trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'RoleCIDA: {total/1e6:.2f}M total | {trainable/1e6:.2f}M trainable')
del test_model, test_x, test_out

## 4. Datasets

### Stage 1: Аргументационные данные
- **Kialo-style через SNLI** — entailment=Prosecutor, contradiction=Defender, neutral=Skeptic
  Это структурно корректно: entailment = 'аргумент принят', contradiction = 'аргумент отвергнут'
- **ArgQ (IBM)** — сила аргумента (0-1). Слабые аргументы → Skeptic

### Stage 2: Неопределённость
- **ChaosNLI** — 100 аннотаторов на каждый пример. Агенты учатся на распределении мнений людей.

### Stage 3: Cross-domain
- NLI + числовые задачи + короткие факты = одинаковая роль в разных доменах

In [ ]:
print('Loading Stage 1: Argument data...')

# SNLI как аргументационный датасет
# label 0=entailment (Prosecutor), 1=neutral (Skeptic), 2=contradiction (Defender)
snli = load_dataset('snli', split='train').filter(lambda x: x['label'] != -1)
snli = snli.map(lambda x: {
    'premise':    x['premise'],
    'hypothesis': x['hypothesis'],
    'text':       x['premise'] + ' | ' + x['hypothesis'],
    'arg_role':   int(x['label']),  # 0=pro, 1=uncertain, 2=con
}, remove_columns=snli.column_names, num_proc=2)
snli = snli.select(range(min(80000, len(snli))))
print(f'SNLI (as argument data): {len(snli):,}')

# MultiNLI для разнообразия жанров
mnli = load_dataset('multi_nli', split='train').filter(lambda x: x['label'] != -1)
mnli = mnli.map(lambda x: {
    'text':     x['premise'] + ' | ' + x['hypothesis'],
    'arg_role': int(x['label']),
}, remove_columns=mnli.column_names, num_proc=2)
mnli = mnli.select(range(min(80000, len(mnli))))
print(f'MNLI (as argument data): {len(mnli):,}')

print('\nLoading Stage 2: ChaosNLI (human annotation distributions)...')
try:
    # ChaosNLI: each example has distribution over 100 annotators
    chaos = load_dataset('chaos_nli', 'snli', split='test')
    HAS_CHAOS = True
    print(f'ChaosNLI: {len(chaos):,} examples with annotator distributions')
    print(f'Features: {list(chaos.features.keys())}')
except Exception as e:
    print(f'ChaosNLI unavailable ({e}), using SNLI with soft labels as substitute')
    HAS_CHAOS = False

print('\nLoading Stage 3: Cross-domain mix...')
ag_news = load_dataset('ag_news', split='train')
ag_news = ag_news.map(lambda x: {
    'text': x['text'], 'arg_role': x['label'] % 3  # map to 3 roles
}, remove_columns=ag_news.column_names, num_proc=2)
ag_news = ag_news.select(range(20000))
print(f'AG News (cross-domain): {len(ag_news):,}')

In [ ]:
# ── Pre-compute all embeddings (fast with frozen MiniLM) ─────────────────────
print('Pre-computing embeddings (this is fast, MiniLM is small)...')

from datasets import concatenate_datasets

# Stage 1 data
arg_data = concatenate_datasets([snli, mnli]).shuffle(seed=42)

print(f'Computing {len(arg_data):,} Stage 1 embeddings...')
t0 = time.time()
arg_embs   = embed_texts(arg_data['text'], batch_size=512)
arg_roles  = torch.tensor(arg_data['arg_role'], dtype=torch.long)
print(f'Done in {time.time()-t0:.0f}s | Shape: {arg_embs.shape}')

# Stage 3 cross-domain
print(f'Computing {len(ag_news):,} Stage 3 embeddings...')
cd_embs  = embed_texts(ag_news['text'], batch_size=512)
cd_roles = torch.tensor(ag_news['arg_role'], dtype=torch.long)
print(f'Done | Shape: {cd_embs.shape}')


class EmbeddingDataset(Dataset):
    """Dataset of pre-computed embeddings. No tokenization at training time."""
    def __init__(self, embs, roles):
        self.embs  = embs    # (N, d)
        self.roles = roles   # (N,) — 0=pro/prosecutor, 1=uncertain/skeptic, 2=con/defender

    def __len__(self): return len(self.roles)

    def __getitem__(self, idx):
        return {'emb': self.embs[idx], 'role': self.roles[idx]}


def make_loaders(embs, roles, val_ratio=0.05, bs=256):
    n  = len(roles); nv = max(500, int(n*val_ratio))
    idx = torch.randperm(n)
    tr_idx, vl_idx = idx[nv:], idx[:nv]
    tr = EmbeddingDataset(embs[tr_idx], roles[tr_idx])
    vl = EmbeddingDataset(embs[vl_idx], roles[vl_idx])
    kw = dict(batch_size=bs, num_workers=4, pin_memory=True,
              persistent_workers=True)
    return DataLoader(tr, shuffle=True, **kw), DataLoader(vl, shuffle=False, **kw)


BS = 256 if DEVICE=='cuda' else 64
s1_trl, s1_vll = make_loaders(arg_embs, arg_roles, bs=BS)
s3_trl, s3_vll = make_loaders(cd_embs,  cd_roles,  bs=BS)
print(f'\nStage 1: {len(s1_trl.dataset):,} train | {len(s1_vll.dataset):,} val')
print(f'Stage 3: {len(s3_trl.dataset):,} train | {len(s3_vll.dataset):,} val')

## 5. Role-Specific Losses

Это сердце всего подхода. Лосс не говорит агенту 'угадай класс'.
Лосс говорит агенту 'веди себя как твоя роль требует'.

```
arg_role=0 (pro/entailment):
    Prosecutor → максимально уверен в '0' (высокая confidence)
    Defender   → должен сомневаться (низкая confidence на '0')
    Skeptic    → максимальная энтропия
    Integrator → следует данным

arg_role=2 (con/contradiction):
    Prosecutor → должен ошибиться в сторону уверенности (это его природа)
    Defender   → максимально уверен что '0' неверно
    Skeptic    → максимальная энтропия
    Integrator → следует данным
```

Плюс **AnchorDiversityLoss** — якоря ролей должны быть ортогональны.

In [ ]:
class RolePreTrainLoss(nn.Module):
    """
    Role-behavioral loss. Teaches agents WHAT TO DO, not WHAT TO ANSWER.
    """
    def __init__(self, n_classes=3):
        super().__init__()
        self.K = n_classes

    def forward(self, b_all, role_anchored, arg_roles, stage, anneal=1.0):
        """
        b_all:        list of (B, M, K) — beliefs per round
        role_anchored:(B, M, d) — role-anchored representations
        arg_roles:    (B,) — 0=pro, 1=uncertain, 2=con
        stage:        1, 2, or 3
        anneal:       0→1 over stage
        """
        if not b_all:
            return torch.tensor(0.0, requires_grad=True), {}

        eps   = 1e-7
        B     = arg_roles.size(0)
        K     = self.K
        device = arg_roles.device

        # Masks per role type in batch
        pro_mask  = (arg_roles == 0)  # entailment / strong pro
        unc_mask  = (arg_roles == 1)  # neutral / uncertain
        con_mask  = (arg_roles == 2)  # contradiction / strong con

        total_loss  = torch.tensor(0.0, device=device)
        info        = defaultdict(float)
        max_ent     = math.log(K)

        # Process each round (weight later rounds more)
        round_w = torch.linspace(0.4, 1.0, len(b_all), device=device)
        round_w = round_w / round_w.sum()

        for rnd, (b_t, w_r) in enumerate(zip(b_all, round_w)):
            _, M, _ = b_t.shape
            b_t = b_t[:, :, :K].clamp(eps, 1-eps)
            b_t = b_t / b_t.sum(-1, keepdim=True)

            agent_losses = []

            for i in range(min(M, 4)):
                b_i   = b_t[:, i, :]  # (B, K)
                conf  = b_i.max(dim=1).values          # (B,)
                ent   = -(b_i * b_i.log()).sum(dim=1)  # (B,)
                b_pro = b_i[:, 0]  # prob of 'pro' class
                b_con = b_i[:, min(2, K-1)]  # prob of 'con' class

                if i == 0:  # ── PROSECUTOR ──────────────────────────────────
                    # On PRO examples: be maximally confident
                    l_pro = -torch.log(b_pro[pro_mask] + eps).mean() if pro_mask.any() else torch.tensor(0.0, device=device)
                    # On CON examples: still lean toward pro (prosecutor bias)
                    # but penalize if too certain about 'con'
                    l_con = torch.clamp(b_con[con_mask] - 0.5, min=0).mean() if con_mask.any() else torch.tensor(0.0, device=device)
                    # On UNCERTAIN: push toward pro (optimistic bias)
                    l_unc = -torch.log(b_pro[unc_mask] + eps).mean() * 0.3 if unc_mask.any() else torch.tensor(0.0, device=device)
                    loss_i = l_pro + 0.3*l_con + 0.2*l_unc
                    info['prosecutor'] += loss_i.item()

                elif i == 1:  # ── DEFENDER ──────────────────────────────────
                    # On CON examples: be maximally confident about 'con'
                    l_con = -torch.log(b_con[con_mask] + eps).mean() if con_mask.any() else torch.tensor(0.0, device=device)
                    # On PRO examples: question the pro claim
                    l_pro = torch.clamp(b_pro[pro_mask] - 0.5, min=0).mean() if pro_mask.any() else torch.tensor(0.0, device=device)
                    # On UNCERTAIN: lean toward skepticism
                    l_unc = -torch.log(b_con[unc_mask] + eps).mean() * 0.3 if unc_mask.any() else torch.tensor(0.0, device=device)
                    loss_i = l_con + 0.3*l_pro + 0.2*l_unc
                    info['defender'] += loss_i.item()

                elif i == 2:  # ── SKEPTIC ────────────────────────────────────
                    # On UNCERTAIN: maximize entropy (be maximally undecided)
                    l_unc = -ent[unc_mask].mean() / max_ent if unc_mask.any() else torch.tensor(0.0, device=device)
                    # On CERTAIN (pro/con): still don't be too confident
                    certain_mask = ~unc_mask
                    l_cert = torch.clamp(conf[certain_mask] - 0.7, min=0).mean() if certain_mask.any() else torch.tensor(0.0, device=device)
                    loss_i = l_unc + 0.4*l_cert
                    info['skeptic'] += loss_i.item()

                else:  # ── INTEGRATOR ────────────────────────────────────────
                    # Follow the data: cross-entropy with true labels
                    y_c = arg_roles.clamp(0, K-1)
                    loss_i = F.nll_loss(b_i.log(), y_c)
                    info['integrator'] += loss_i.item()

                agent_losses.append(loss_i)

            # ── ANCHOR DIVERSITY LOSS ──────────────────────────────────────────
            # Role representations should be orthogonal to each other
            r_norm = F.normalize(role_anchored.mean(0), p=2, dim=-1)  # (M, d)
            gram   = r_norm @ r_norm.T  # (M, M)
            eye    = torch.eye(M, device=device)
            anchor_div = ((gram - eye)**2).mean()
            info['anchor_div'] += anchor_div.item()

            # ── CONSENSUS COHERENCE LOSS ──────────────────────────────────────
            # Integrator should be closest to consensus of all agents
            b_mean = b_t.mean(dim=1, keepdim=True)  # (B, 1, K)
            integrator_b = b_t[:, 3:4, :]  # (B, 1, K)
            coherence = F.kl_div(
                (integrator_b + eps).log(), b_mean + eps,
                reduction='batchmean'
            )
            info['coherence'] += coherence.item()

            round_total = (torch.stack(agent_losses).mean()
                          + 0.1 * anchor_div
                          + 0.05 * coherence * anneal)
            total_loss  = total_loss + w_r * round_total

        n_rounds = len(b_all)
        return total_loss, {k: v/n_rounds for k,v in info.items()}


# Stage 2: Uncertainty calibration loss
class UncertaintyCalibrationLoss(nn.Module):
    """
    Stage 2: align agent beliefs with human annotation distributions.
    If ChaosNLI unavailable: use soft label approximation.
    """
    def forward(self, b_all, arg_roles, soft_labels=None):
        if not b_all: return torch.tensor(0.0, requires_grad=True), {}
        eps = 1e-7
        K   = b_all[0].shape[-1]

        # If no soft labels: create them from hard labels with noise
        if soft_labels is None:
            y_oh   = F.one_hot(arg_roles.clamp(0, K-1), K).float()
            # Simulate human disagreement: smooth labels
            alpha  = 0.7  # confidence of majority
            soft   = alpha * y_oh + (1-alpha)/K
        else:
            soft = soft_labels  # (B, K) real distributions

        b_final = b_all[-1][:, :, :K].clamp(eps, 1-eps)
        b_final = b_final / b_final.sum(-1, keepdim=True)

        losses = []
        for i in range(b_final.size(1)):
            b_i = b_final[:, i, :]
            if i == 2:  # Skeptic: match human disagreement distribution exactly
                kl = F.kl_div((b_i+eps).log(), soft+eps, reduction='batchmean')
                losses.append(kl)
            elif i == 3:  # Integrator: also match distribution
                kl = F.kl_div((b_i+eps).log(), soft+eps, reduction='batchmean')
                losses.append(kl * 0.8)
            else:  # Prosecutor/Defender: standard role losses still apply
                ce = F.nll_loss(b_i.log(), arg_roles.clamp(0, K-1))
                losses.append(ce * 0.5)

        loss = torch.stack(losses).mean()
        return loss, {'calib': loss.item()}


role_loss_fn  = RolePreTrainLoss(n_classes=3)
calib_loss_fn = UncertaintyCalibrationLoss()
print('Role losses defined')

## 6. Training Engine

In [ ]:
def compute_asi(b_all):
    if not b_all: return 0.0
    b = b_all[-1]
    ents = [-(b[:,ag,:].clamp(1e-7,1)*b[:,ag,:].clamp(1e-7,1).log()).sum(1).mean().item()
            for ag in range(b.shape[1])]
    return float(np.std(ents))


@torch.no_grad()
def evaluate(model, loader, max_batches=80):
    model.eval()
    all_acc, all_dis, all_asi = [], [], []
    # Per-agent behavior stats
    agent_conf = defaultdict(list)
    agent_ent  = defaultdict(list)

    for i, batch in enumerate(loader):
        if i >= max_batches: break
        x = batch['emb'].to(DEVICE)
        y = batch['role'].to(DEVICE).clamp(0, 2)

        with torch.autocast(device_type='cuda' if DEVICE=='cuda' else 'cpu',
                            dtype=DTYPE, enabled=IS_AMP):
            out = model(x)

        preds = out['p_final'][:, :3].argmax(1)
        all_acc.append((preds == y).float().mean().item())
        all_dis.append(out['disagreement'].mean().item())
        all_asi.append(compute_asi(out['b_all']))

        if out['b_all']:
            b = out['b_all'][-1]
            for ag in range(b.shape[1]):
                bi = b[:, ag, :].clamp(1e-7, 1)
                agent_conf[ag].append(bi.max(1).values.mean().item())
                agent_ent[ag].append(-(bi*bi.log()).sum(1).mean().item())

    agents = {ag: {'conf': np.mean(agent_conf[ag]), 'ent': np.mean(agent_ent[ag])}
              for ag in sorted(agent_conf)}
    return {
        'acc':          np.mean(all_acc),
        'disagreement': np.mean(all_dis),
        'asi':          np.mean(all_asi),
        'agents':       agents,
    }


def train_epoch(model, loader, optimizer, scheduler, stage, anneal):
    model.train()
    total_loss = 0.0; n = 0
    scaler = torch.cuda.amp.GradScaler(enabled=(DTYPE==torch.float16))

    for batch in loader:
        x = batch['emb'].to(DEVICE, non_blocking=True)
        y = batch['role'].to(DEVICE, non_blocking=True).clamp(0, 2)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda' if DEVICE=='cuda' else 'cpu',
                            dtype=DTYPE, enabled=IS_AMP):
            out = model(x)

            if stage in [1, 3]:
                loss, info = role_loss_fn(
                    out['b_all'], out['role_anchored'], y, stage, anneal
                )
            else:  # stage 2: calibration
                loss_r, _ = role_loss_fn(
                    out['b_all'], out['role_anchored'], y, 1, anneal
                )
                loss_c, _ = calib_loss_fn(out['b_all'], y)
                loss = 0.5 * loss_r + 0.5 * loss_c
                info = {'combined': loss.item()}

            if out.get('L_orth') is not None:
                loss = loss + 0.02 * out['L_orth']

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item(); n += 1

    return total_loss / max(n, 1)


class EarlyStopping:
    def __init__(self, patience=3, metric='asi', mode='max'):
        self.patience=patience; self.metric=metric; self.mode=mode
        self.best=float('-inf') if mode=='max' else float('inf')
        self.counter=0
    def __call__(self, ev):
        val = ev[self.metric]
        better = val > self.best if self.mode=='max' else val < self.best
        if better:
            self.best=val; self.counter=0; return False
        self.counter+=1
        return self.counter >= self.patience

print('Training engine OK')

## 7. Optimizer

In [ ]:
STAGE_EPOCHS = {1: 6, 2: 5, 3: 5}  # Total: 16 epochs

# Stage 1: role identity + CIDA
# Stage 3 uses same model, so build once
model_s1 = RoleCIDA(num_classes=3).to(DEVICE)

# Separate LRs: role anchors get higher LR (they need to diverge fast)
optimizer = optim.AdamW([
    {'params': model_s1.role_layer.parameters(), 'lr': 5e-4, 'weight_decay': 0.0},
    {'params': model_s1.plugin.parameters(),     'lr': 2e-4, 'weight_decay': 1e-2},
], fused=(DEVICE=='cuda'))

total_steps = sum(
    STAGE_EPOCHS[s] * len(s1_trl if s != 3 else s3_trl)
    for s in [1, 2, 3]
)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[8e-4, 3e-4],
    total_steps=total_steps,
    pct_start=0.05,
    anneal_strategy='cos',
)

print(f'Total steps: {total_steps:,}')
print(f'Est. time: ~{total_steps*0.03/3600:.1f}h on A100')

## 8. Pre-Training Loop

Early stopping следит за **ASI** (Agent Specialization Index), а не accuracy.
Цель не угадать класс — цель максимизировать специализацию агентов.

In [ ]:
best_asi   = 0.0
global_ep  = 0
stage_loaders = {1: s1_trl, 2: s1_trl, 3: s3_trl}
stage_val     = {1: s1_vll, 2: s1_vll, 3: s3_vll}

for stage in [1, 2, 3]:
    stage_names = {
        1: 'Role Identity (argument data)',
        2: 'Uncertainty Calibration',
        3: 'Cross-Domain Generalization',
    }
    print(f'\n{"="*60}')
    print(f'  STAGE {stage}: {stage_names[stage]}')
    print(f'{"="*60}')

    # Early stopping on ASI для стадий 1 и 2
    # На стадии 3 — на disagreement (хотим чтобы кросс-доменно сохранялась дивергенция)
    es_metric = 'asi' if stage < 3 else 'disagreement'
    stage_es  = EarlyStopping(patience=3, metric=es_metric, mode='max')
    loader_tr = stage_loaders[stage]
    loader_vl = stage_val[stage]

    for ep in range(STAGE_EPOCHS[stage]):
        global_ep  += 1
        anneal      = ep / max(STAGE_EPOCHS[stage]-1, 1)
        t0          = time.time()

        avg_loss = train_epoch(model_s1, loader_tr, optimizer, scheduler, stage, anneal)
        ev       = evaluate(model_s1, loader_vl, max_batches=80)
        dt       = time.time() - t0

        row = {'stage':stage,'epoch':global_ep,'loss':avg_loss,
               'time':dt, **{k:v for k,v in ev.items() if k!='agents'}}
        HISTORY.append(row)

        print(f'\nEp {global_ep}/{sum(STAGE_EPOCHS.values())} [{dt:.0f}s] '
              f'loss={avg_loss:.4f} acc={ev["acc"]:.4f} '
              f'dis={ev["disagreement"]:.4f} ASI={ev["asi"]:.4f}')

        # Per-agent behavior
        print('  Agent behavior:')
        for ag, s in ev['agents'].items():
            name = RoleNames.get(ag)
            print(f'    {name:<12}: conf={s["conf"]:.3f} ent={s["ent"]:.3f}')

        # Anchor similarity (should decrease)
        sim = model_s1.role_layer.anchor_similarity_matrix()
        off_diag = sim[~torch.eye(4, dtype=bool)].abs().mean().item()
        print(f'  Anchor off-diag similarity: {off_diag:.4f} (lower=more diverse)')

        # Save best checkpoint (by ASI)
        if ev['asi'] > best_asi:
            best_asi = ev['asi']
            model_s1.plugin.save_pretrained('ckpt_role/cida_role_best')
            torch.save({
                'plugin':     model_s1.plugin.state_dict(),
                'role_layer': model_s1.role_layer.state_dict(),
                'config':     model_s1.plugin.config.to_dict(),
                'history':    HISTORY,
            }, 'ckpt_role/cida_role_best_full.pt')
            print(f'  ✓ Best checkpoint (ASI={best_asi:.4f})')

        # Stage checkpoint
        model_s1.plugin.save_pretrained(f'ckpt_role/cida_stage{stage}_ep{ep+1}')

        if stage_es(ev):
            print(f'  Early stop: {es_metric} not improving')
            break

    model_s1.plugin.save_pretrained(f'ckpt_role/cida_stage{stage}_final')
    print(f'Stage {stage} done. Best ASI so far: {best_asi:.4f}')

print(f'\nRole pre-training complete!')
print(f'Best ASI: {best_asi:.4f}')

## 9. Role Analysis — Что выучили агенты

In [ ]:
model_s1.eval()

# 1. Anchor diversity evolution не можем показать (нет истории),
#    но можем показать финальные якоря
print('=== FINAL ANCHOR SIMILARITY MATRIX ===')
print('(Diagonal=1.0, off-diagonal should be LOW)')
sim = model_s1.role_layer.anchor_similarity_matrix()
print(sim.numpy().round(3))
off = sim[~torch.eye(4, dtype=bool)].abs().mean().item()
print(f'Mean off-diagonal similarity: {off:.4f}')
print(f'-> {'GOOD (diverse anchors)' if off < 0.3 else 'NEEDS MORE TRAINING'}')

# 2. Per-role behavior на разных типах примеров
print('\n=== BEHAVIOR BY ARGUMENT TYPE ===')
results_by_type = {}
with torch.no_grad():
    sample = next(iter(s1_vll))
    x = sample['emb'].to(DEVICE)
    y = sample['role'].to(DEVICE)
    out = model_s1(x)

    if out['b_all']:
        b = out['b_all'][-1]  # (B, M, K)
        for role_type, mask_val, name in [(0,'PRO'),(1,'UNCERTAIN'),(2,'CON')]:
            mask = (y == role_type)
            if not mask.any(): continue
            print(f'\n  Input type: {name} (n={mask.sum().item()})')
            for ag in range(4):
                bi   = b[mask, ag, :]
                conf = bi.max(1).values.mean().item()
                ent  = -(bi.clamp(1e-7,1)*bi.clamp(1e-7,1).log()).sum(1).mean().item()
                print(f'    {RoleNames.get(ag):<12}: conf={conf:.3f} ent={ent:.3f}')

print('\nExpected behavior:')
print('  PRO input:       Prosecutor=HIGH conf, Defender=LOW conf, Skeptic=HIGH ent')
print('  UNCERTAIN input: All agents=HIGH ent (especially Skeptic)')
print('  CON input:       Defender=HIGH conf, Prosecutor=LOW conf, Skeptic=HIGH ent')

## 10. Visualization

In [ ]:
if not HISTORY:
    print('No history yet.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    sc = {1: '#EF4444', 2: '#3B82F6', 3: '#10B981'}
    ep_r = [h['epoch'] for h in HISTORY]

    def pm(ax, key, title, ylabel=''):
        vals = [h.get(key, 0) for h in HISTORY]
        ax.plot(ep_r, vals, 'o-', color='#1E293B', lw=2)
        for e, v, h in zip(ep_r, vals, HISTORY):
            ax.scatter(e, v, color=sc[h['stage']], s=90, zorder=5)
        ax.set_title(title, fontweight='bold')
        ax.set_ylabel(ylabel); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)

    pm(axes[0,0], 'loss',          'Training Loss',              'Loss')
    pm(axes[0,1], 'asi',           'ASI — Agent Specialization', 'ASI')
    pm(axes[0,2], 'disagreement',  'Agent Disagreement',         'Dis')
    pm(axes[1,0], 'acc',           'Task Accuracy (proxy)',      'Acc')

    # Epoch times
    ax = axes[1,1]
    times = [h['time']/60 for h in HISTORY]
    ax.bar(ep_r, times, color=[sc[h['stage']] for h in HISTORY], alpha=0.85)
    ax.axhline(np.mean(times), color='red', ls='--',
               label=f'Avg={np.mean(times):.1f}min')
    ax.set_title('Epoch Time (min)', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)

    # Stage breakdown
    ax = axes[1,2]
    for s, c in sc.items():
        hs = [h for h in HISTORY if h['stage']==s]
        if hs:
            best_h = max(hs, key=lambda h: h['asi'])
            x_pos = s - 1
            ax.bar(x_pos-0.2, best_h['asi'],          0.35, color=c, label=f'S{s} ASI',  alpha=0.85)
            ax.bar(x_pos+0.2, best_h['disagreement'], 0.35, color=c, label=f'S{s} Dis',  alpha=0.5)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(['Stage1','Stage2','Stage3'])
    ax.set_title('Best ASI & Disagreement per Stage', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    for s, c in sc.items():
        axes[0,0].scatter([], [], color=c, label=f'Stage {s}', s=80)
    axes[0,0].legend()

    plt.suptitle('CIDA Role Pre-Training — Agent Identity Learning',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results_role/role_pretrain.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Best ASI: {best_asi:.4f}')

## 11. Transfer Test — Работает ли роль на новом домене?

Берём предобученный CIDA и проверяем: сохраняется ли ролевое поведение
когда мы показываем ему данные из другого домена (AG News — новости).
Новый энкодер не нужен — MiniLM универсален.

**Критерий успеха:** ASI на новом домене ≥ 80% от ASI на обучающем домене.

In [ ]:
print('Transfer test: role behavior on AG News (unseen domain)...')

model_s1.eval()
with torch.no_grad():
    # Sample from test domain
    x_new = cd_embs[:500].to(DEVICE)
    y_new = cd_roles[:500].to(DEVICE)
    out_new = model_s1(x_new)

# Compute ASI on new domain
asi_new = compute_asi(out_new['b_all'])
dis_new = out_new['disagreement'].mean().item()

# Compare to training domain
ev_train = evaluate(model_s1, s1_vll, max_batches=20)

print(f'\nASI:          Train={ev_train["asi"]:.4f} | Transfer={asi_new:.4f} | '
      f'Ratio={asi_new/max(ev_train["asi"],1e-6):.2f}')
print(f'Disagreement: Train={ev_train["disagreement"]:.4f} | Transfer={dis_new:.4f}')

if ev_train['asi'] > 0:
    ratio = asi_new / ev_train['asi']
    if ratio >= 0.8:
        print(f'\n✓ TRANSFER SUCCESS: roles preserved at {ratio*100:.0f}% on new domain')
        print('  Agents behave according to role regardless of domain.')
    else:
        print(f'\n⚠ Partial transfer: {ratio*100:.0f}% — consider more Stage 3 epochs')

# Per-agent on new domain
if out_new['b_all']:
    b = out_new['b_all'][-1]
    print('\nPer-agent behavior on AG News:')
    for ag in range(4):
        bi   = b[:, ag, :].clamp(1e-7, 1)
        conf = bi.max(1).values.mean().item()
        ent  = -(bi*bi.log()).sum(1).mean().item()
        print(f'  {RoleNames.get(ag):<12}: conf={conf:.3f} ent={ent:.3f}')

## 12. Summary & Next Steps

In [ ]:
print('='*65)
print('CIDA ROLE PRE-TRAINING SUMMARY')
print('='*65)
print(f'Best ASI achieved: {best_asi:.4f}')
print(f'Final anchor diversity: {off:.4f} off-diagonal similarity')
if HISTORY:
    avg_t = np.mean([h['time'] for h in HISTORY])
    print(f'Avg epoch time: {avg_t:.0f}s')

print()
print('Saved checkpoints:')
print('  ckpt_role/cida_role_best/          <- plugin only (HF format)')
print('  ckpt_role/cida_role_best_full.pt   <- plugin + role_layer + config')
print()
print('How to use in downstream task:')
print()
print('  # Load pretrained CIDA')
print('  plugin = CIDAPlugin.from_pretrained("ckpt_role/cida_role_best")')
print()
print('  # Plug into ANY encoder')
print('  out = plugin(densenet_features)    # CheXpert')
print('  out = plugin(bert_cls_token)       # NLP')
print('  out = plugin(whisper_embedding)    # Voice')
print('  out = plugin(resnet_pooled)        # General CV')
print()
print('  # Agents already know their roles.')
print('  # Only need to fine-tune the task head.')